# HANNA Benchmark 演示 Notebook



这部分展示的是我们项目里的 benchmark 实现。它的来源是 HANNA，但不是直接复现 HANNA 官方模型，而是把 HANNA 的六个评价维度接入到我们自己的项目里，做成一个可以运行的 Judge。



HANNA 本身解决的是故事生成任务里“怎么评”的问题。它给出了一套比较清楚的评价标准，把故事质量拆成六个维度：相关性、连贯性、共情力、惊喜度、吸引力和复杂度。仓库里提供了数据、人工标注结果和分析代码，但没有直接提供一个可下载的评分模型。



我们当前的做法，是复用这套六维标准，用现有大模型完成评分，并把输出收束成结构化结果。下面先从代码入口开始，说明这套 benchmark 在项目里已经落成了实际模块。

In [ ]:
from pathlib import Path
import sys
import json

project_root = Path.cwd()
if project_root.name == 'benchmark':
    project_root = project_root.parent

demo_dir = project_root / 'demo'
sys.path.insert(0, str(demo_dir))
sys.path.insert(0, str(demo_dir / 'src'))

from src.story_judge import StoryJudge, format_judgement_for_cli

print('Demo directory:', demo_dir)
print('StoryJudge import success')

## 2. 输入形式



这套评分方式最少只需要一段故事文本就可以运行；如果再补上 prompt 和 reference，相关性判断会更完整一些。



因此，当前 benchmark 的输入可以分成三层：



- 故事文本：待评分对象

- prompt：故事原本对应的题目或生成要求

- reference：原始素材、参考文本或回忆录内容



下面这个样例把这三部分摆出来，方便直接看到评分器接收的输入长什么样。

In [ ]:
sample_story = '''那年冬天，我和父亲冒着雪去镇上卖菜。路很滑，他一路扶着车不说话，只在集市散去时给我买了一只热乎乎的烤红薯。很多年后我才明白，那天他不是不累，只是舍不得让我看出来。'''
sample_prompt = '写一个关于父爱与冬日记忆的短故事'
sample_reference = '父亲在寒冷冬天默默照顾孩子，很多爱没有直接说出口，而是藏在具体行动里。'

print(sample_story)

## 3. 六维评分



评分的核心动作发生在这里。Judge 会同时读取故事、prompt 和 reference，然后按六个维度分别打分，并生成一个总评。



这六个维度分别是：



- Relevance：故事和题目或参考文本是否贴合

- Coherence：故事是否顺、逻辑是否通

- Empathy：人物情绪能不能被感受到

- Surprise：有没有合理的转折和意外感

- Engagement：读起来是否有吸引力

- Complexity：内容是否足够丰富，不显得单薄



输出结果不是一句泛泛而谈的评价，而是一份结构化 JSON，里面包含六维分数、每维解释和总分。

In [ ]:
judge = StoryJudge()
result = judge.judge_story(
    story=sample_story,
    story_prompt=sample_prompt,
    reference_story=sample_reference,
)

print(format_judgement_for_cli(result))

## 4. 封装成函数



单篇故事评分只是第一步。真正让 benchmark 变得可用的，是把评分逻辑稳定封装下来，这样同一套标准就可以反复用于不同故事、不同版本和不同实验结果。



下面这一段把评分动作包成函数。这样后面不管是做小规模测试，还是做批量评测，都可以直接复用同一个入口。

In [ ]:
def score_story(story, prompt=None, reference=None):
    judge = StoryJudge()
    result = judge.judge_story(
        story=story,
        story_prompt=prompt,
        reference_story=reference,
    )
    return result

demo_result = score_story(
    story='我一直以为母亲只是节俭，后来才知道她把最好的都留给了我们。',
    prompt='写一个关于母爱的短故事'
)

demo_result['scores']

## 5. 用同一套口径比较不同结果



benchmark 的价值不只是在一篇故事上给出一个分数，更重要的是把不同结果放到同一把尺子下比较。



当后面的问题生成、对话策略或者故事生成方式发生变化时，只要继续用这套六维标准去评，就能比较清楚地看到：哪一版更连贯，哪一版更有情感，哪一版和原始素材更贴近。



下面这段代码演示的是最简单的批量比较形式：给多篇故事打分，再把结果整理到统一结构里。

In [ ]:
stories = [
    {
        'name': 'story_a',
        'story': '小时候我总觉得父亲严厉，直到离家后才明白那些沉默背后都是牵挂。',
        'prompt': '写一个关于父爱的短故事'
    },
    {
        'name': 'story_b',
        'story': '那天的晚饭很简单，母亲把唯一的鸡腿夹到我碗里，自己却说不饿。',
        'prompt': '写一个关于母爱的短故事'
    }
]

batch_results = []
for item in stories:
    scored = score_story(item['story'], prompt=item.get('prompt'))
    batch_results.append({
        'name': item['name'],
        'final_score': scored['final_score'],
        **scored['scores']
    })

batch_results

## 6. 这部分 benchmark 的定位



这套 benchmark 在项目里的定位很明确：它提供了一套统一的评价口径，用来评估生成故事的质量，并支持后续版本之间的比较。



它的基础来自 HANNA 六维标准，但实现方式是我们自己的 Judge。也就是说，当前用的不是 HANNA 官方发布的专门模型，而是把 HANNA 的评价思想落到了现有工程代码里。



它已经能完成三件事：



- 对单篇故事做六维评分

- 输出结构化结果，便于保存和展示

- 对不同版本的故事做统一比较



同时，这一版也有边界。它本质上仍然是 LLM Judge，不是专门训练出来的 evaluator，所以更适合做相对比较和迭代评估，而不适合把某一次单独分数直接当成绝对结论。



如果后续继续往前推进，比较自然的方向就是：增加多次采样来降低波动，用中文人工标注数据做校准，在数据量足够时再训练专门的 evaluator。